# 04 - Modelagem da Camada Gold

## 1. Objetivo

A camada Gold representa a camada de consumo analítico da arquitetura Medallion
implementada neste projeto.

Enquanto a camada Bronze preserva os dados provenientes das fontes e a camada
Silver realiza padronização, tratamento e validação da qualidade, a camada Gold
organiza os dados de acordo com necessidades analíticas e regras de negócio.

Foi adotada uma abordagem de modelagem dimensional, separando dimensões
descritivas e tabelas fato. Essa estrutura facilita consultas analíticas,
agregações e a utilização dos dados por ferramentas de Business Intelligence.

A modelagem foi desenvolvida para permitir análises relacionadas a:

- evolução das vendas e dos pedidos;
- faturamento e ticket médio;
- produtos e categorias comercializadas;
- desempenho dos vendedores;
- distribuição geográfica dos clientes;
- custos de frete;
- meios e condições de pagamento;
- desempenho logístico e atrasos de entrega;
- avaliações e satisfação dos clientes.

## 2. Estratégia de Modelagem

A camada Gold utiliza conceitos de modelagem dimensional, com definição explícita
da granularidade das tabelas fato.

Foram consideradas três granularidades principais:

**Pedido:** uma linha representa um pedido realizado pelo cliente.

**Item do pedido:** uma linha representa um produto comercializado dentro de
um pedido, permitindo análises por produto, categoria e vendedor.

**Pagamento:** uma linha representa uma ocorrência de pagamento associada a
um pedido.

As dimensões concentram atributos descritivos utilizados para segmentação e
contextualização das métricas.

A separação entre diferentes granularidades evita duplicações de métricas. Por
exemplo, pagamentos não são incorporados diretamente à fato de itens, pois um
pedido pode possuir múltiplos itens e múltiplos pagamentos, o que poderia gerar
relações muitos-para-muitos e duplicação de valores.

Modelo Dimensional Proposto:

_DIMENSÕES:_
1) dim_customer
2) dim_product
3) dim_seller
4) dim_date
5) dim_order

_FATOS:_
1) fact_order_items
2) fact_payments


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [ ]:
CATALOG = "datalake_mvp"
SILVER = "mvp_silver"
GOLD = "mvp_gold"

print(f"Catálogo: {CATALOG}")
print(f"Origem:   {CATALOG}.{SILVER}")
print(f"Destino:  {CATALOG}.{GOLD}")

In [ ]:
def silver_table(nome):
    return spark.table(f"{CATALOG}.{SILVER}.{nome}")


def salvar_gold(df, nome):
    tabela_destino = f"{CATALOG}.{GOLD}.{nome}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )

    print(
        f"✓ {tabela_destino} criada com "
        f"{df.count():,} registros"
    )

In [ ]:
customers = silver_table("customers")
sellers = silver_table("sellers")
products = silver_table("products")
orders = silver_table("orders")
items = silver_table("order_items")
payments = silver_table("order_payments")
reviews = silver_table("order_reviews")
geolocation = silver_table("geolocation")

print("✓ Tabelas Silver carregadas com sucesso.")

## 3.1 - Dimensão de Clientes

A dimensão de clientes possui granularidade de uma linha por `customer_id`.

Os dados cadastrais provenientes da camada Silver foram enriquecidos com
informações geográficas consolidadas a partir do prefixo de CEP.

O enriquecimento permite realizar análises geográficas sem a necessidade de
consultar diretamente a tabela original de geolocalização, cuja granularidade
é diferente da entidade cliente.

In [ ]:
dim_customer = (
    customers.alias("c")
    .join(
        geolocation.alias("g"),
        F.col("c.customer_zip_code_prefix")
        == F.col("g.geolocation_zip_code_prefix"),
        "left"
    )
    .select(
        F.col("c.customer_id"),
        F.col("c.customer_unique_id"),

        F.col("c.customer_zip_code_prefix"),

        F.col("c.customer_city"),
        F.col("c.customer_state"),

        F.col("g.geolocation_lat")
        .alias("customer_lat"),

        F.col("g.geolocation_lng")
        .alias("customer_lng")
    )
)

salvar_gold(dim_customer, "dim_customer")

In [ ]:
silver_count = customers.count()
gold_count = dim_customer.count()

print(f"Silver customers : {silver_count:,}")
print(f"Gold dim_customer: {gold_count:,}")
print(f"Diferença        : {silver_count - gold_count:,}")

In [ ]:
display(
    dim_customer.select(
        F.count("*").alias("clientes"),

        F.sum(
            F.when(
                F.col("customer_lat").isNull(),
                1
            ).otherwise(0)
        ).alias("sem_geolocalizacao")
    )
)

### Resultado da transformação

A dimensão `dim_customer` foi criada com 99.441 registros, mantendo integralmente
a granularidade da tabela de clientes da camada Silver.

O enriquecimento geográfico identificou coordenadas para a maior parte dos
clientes. Apenas 278 registros não apresentaram correspondência com a base
consolidada de geolocalização, representando aproximadamente 0,28% dos clientes.

Esses registros foram preservados na dimensão, mantendo latitude e longitude
como valores nulos. A ausência de correspondência geográfica não foi considerada
motivo suficiente para exclusão do cliente, evitando perda de informações
transacionais nas análises posteriores.

## 3.2 - Dimensão de Vendedores

A dimensão de vendedores possui granularidade de uma linha por `seller_id`.

Os dados cadastrais da camada Silver foram enriquecidos com latitude e longitude
obtidas a partir da tabela consolidada de geolocalização, utilizando o prefixo
de CEP como chave de associação.

Foi utilizado `LEFT JOIN` para preservar todos os vendedores, inclusive aqueles
sem correspondência geográfica.

In [ ]:
dim_seller = (
    sellers.alias("s")
    .join(
        geolocation.alias("g"),
        F.col("s.seller_zip_code_prefix")
        == F.col("g.geolocation_zip_code_prefix"),
        "left"
    )
    .select(
        F.col("s.seller_id"),
        F.col("s.seller_zip_code_prefix"),
        F.col("s.seller_city"),
        F.col("s.seller_state"),

        F.col("g.geolocation_lat")
        .alias("seller_lat"),

        F.col("g.geolocation_lng")
        .alias("seller_lng")
    )
)

salvar_gold(dim_seller, "dim_seller")

In [ ]:
silver_count = sellers.count()
gold_count = dim_seller.count()

sem_geo_seller = (
    dim_seller
    .filter(F.col("seller_lat").isNull())
    .count()
)

print(f"Silver sellers      : {silver_count:,}")
print(f"Gold dim_seller     : {gold_count:,}")
print(f"Diferença           : {silver_count - gold_count:,}")
print(f"Sem geolocalização  : {sem_geo_seller:,}")

## 3.3 - Dimensão de Produtos

A dimensão de produtos possui granularidade de uma linha por `product_id`.

Foram selecionados atributos descritivos e físicos relevantes para análise.
A tradução da categoria, incorporada anteriormente na camada Silver, foi
preservada para facilitar o consumo analítico.

A dimensão também mantém os atributos físicos dos produtos, possibilitando
análises relacionadas a peso, dimensões, quantidade de imagens e características
descritivas.

In [ ]:
dim_product = (
    products
    .select(
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )
)

salvar_gold(dim_product, "dim_product")

In [ ]:
silver_count = products.count()
gold_count = dim_product.count()

product_distinct = (
    dim_product
    .select("product_id")
    .distinct()
    .count()
)

print(f"Silver products       : {silver_count:,}")
print(f"Gold dim_product      : {gold_count:,}")
print(f"Diferença             : {silver_count - gold_count:,}")
print(f"Product IDs distintos : {product_distinct:,}")

## 3.4 - Dimensão Calendário

A dimensão calendário foi criada para centralizar os atributos temporais utilizados
nas análises da camada Gold.

O intervalo da dimensão é definido dinamicamente a partir da menor e da maior
data de compra existentes na base de pedidos, evitando a definição manual de
períodos.

A dimensão possui granularidade diária, com uma linha para cada data do período,
e inclui atributos derivados como ano, trimestre, mês, semana e dia da semana.

Também foi criada uma chave no formato `yyyyMMdd`, facilitando relacionamentos
com tabelas fato e o consumo dos dados por ferramentas de Business Intelligence.

In [ ]:
intervalo_datas = (
    orders
    .select(
        F.min(
            F.to_date("order_purchase_timestamp")
        ).alias("data_minima"),

        F.max(
            F.to_date("order_purchase_timestamp")
        ).alias("data_maxima")
    )
)

display(intervalo_datas)

In [ ]:
datas = intervalo_datas.first()

data_minima = datas["data_minima"]
data_maxima = datas["data_maxima"]

print(f"Data inicial: {data_minima}")
print(f"Data final  : {data_maxima}")

In [ ]:
dim_date = (
    spark.sql(
        f"""
        SELECT EXPLODE(
            SEQUENCE(
                TO_DATE('{data_minima}'),
                TO_DATE('{data_maxima}'),
                INTERVAL 1 DAY
            )
        ) AS date
        """
    )

    .withColumn(
        "date_key",
        F.date_format("date", "yyyyMMdd").cast("int")
    )

    .withColumn(
        "year",
        F.year("date")
    )

    .withColumn(
        "quarter",
        F.quarter("date")
    )

    .withColumn(
        "month",
        F.month("date")
    )

    .withColumn(
        "month_name",
        F.date_format("date", "MMMM")
    )

    .withColumn(
        "year_month",
        F.date_format("date", "yyyy-MM")
    )

    .withColumn(
        "day",
        F.dayofmonth("date")
    )

    .withColumn(
        "day_of_week",
        F.dayofweek("date")
    )

    .withColumn(
        "day_name",
        F.date_format("date", "EEEE")
    )

    .withColumn(
        "week_of_year",
        F.weekofyear("date")
    )

    .withColumn(
        "is_weekend",
        F.when(
            F.dayofweek("date").isin(1, 7),
            True
        ).otherwise(False)
    )
)

In [ ]:
salvar_gold(dim_date, "dim_date")

In [ ]:
display(
    dim_date
    .orderBy("date")
    .limit(20)
)

In [ ]:
total_datas = dim_date.count()

datas_distintas = (
    dim_date
    .select("date_key")
    .distinct()
    .count()
)

print(f"Registros calendário : {total_datas:,}")
print(f"Chaves distintas     : {datas_distintas:,}")
print(f"Diferença            : {total_datas - datas_distintas:,}")

#4.5 - Construção da Fato de Vendas

A tabela fato é construída na granularidade de item do pedido, utilizando order_id e order_item_id como identificadores do evento de venda.

A estrutura integra informações dos itens com os respectivos pedidos, permitindo relacionar cada transação às dimensões de cliente, produto, vendedor e tempo.

São mantidas as principais métricas quantitativas da operação, como preço do produto, valor do frete e valor total do item.

In [ ]:
fact_order_items = (
    items.alias("i")
    .join(
        orders.alias("o"),
        F.col("i.order_id") == F.col("o.order_id"),
        "left"
    )
    .select(
        # Chaves da fato
        F.col("i.order_id"),
        F.col("i.order_item_id"),

        # Chaves dimensionais
        F.col("o.customer_id"),
        F.col("i.product_id"),
        F.col("i.seller_id"),

        # Datas
        F.col("o.order_purchase_timestamp"),
        F.to_date(
            F.col("o.order_purchase_timestamp")
        ).alias("order_purchase_date"),

        # Informações operacionais
        F.col("o.order_status"),
        F.col("i.shipping_limit_date"),

        # Métricas
        F.col("i.price"),
        F.col("i.freight_value"),
        F.col("i.item_total_value"),

        # Métricas de prazo já tratadas na Silver
        F.col("o.delivery_time_days"),
        F.col("o.delivery_delay_days"),
        F.col("o.flag_late_delivery")
    )
)

print("✓ fact_order_items construída.")

In [ ]:
display(
    fact_order_items
    .select(
        "order_id",
        "order_item_id",
        "customer_id",
        "product_id",
        "seller_id",
        "order_purchase_date",
        "price",
        "freight_value",
        "item_total_value",
        "delivery_time_days",
        "delivery_delay_days",
        "flag_late_delivery"
    )
    .limit(20)
)

In [ ]:
total_fact = fact_order_items.count()

chaves_fact_distintas = (
    fact_order_items
    .select(
        "order_id",
        "order_item_id"
    )
    .distinct()
    .count()
)

print(f"Registros fact          : {total_fact:,}")
print(f"Chaves distintas        : {chaves_fact_distintas:,}")
print(f"Duplicidades            : {total_fact - chaves_fact_distintas:,}")

##4.5.1 - Validação da Integridade Referencial

Antes da persistência da tabela fato, são verificadas as correspondências entre suas chaves dimensionais e as respectivas dimensões da camada Gold.

Essa validação permite identificar registros órfãos e assegurar a consistência dos relacionamentos do modelo dimensional.

In [ ]:
# Chaves disponíveis nas dimensões Gold

customer_keys = (
    dim_customer
    .select("customer_id")
    .distinct()
)

product_keys = (
    dim_product
    .select("product_id")
    .distinct()
)

seller_keys = (
    dim_seller
    .select("seller_id")
    .distinct()
)

date_keys = (
    dim_date
    .select(
        F.col("date").alias("dim_date")
    )
    .distinct()
)

In [ ]:
orfaos_customer = (
    fact_order_items
    .join(
        customer_keys,
        "customer_id",
        "left_anti"
    )
    .count()
)

orfaos_product = (
    fact_order_items
    .join(
        product_keys,
        "product_id",
        "left_anti"
    )
    .count()
)

orfaos_seller = (
    fact_order_items
    .join(
        seller_keys,
        "seller_id",
        "left_anti"
    )
    .count()
)

orfaos_date = (
    fact_order_items.alias("f")
    .join(
        date_keys.alias("d"),
        F.col("f.order_purchase_date") == F.col("d.dim_date"),
        "left_anti"
    )
    .count()
)

print(f"Órfãos customer : {orfaos_customer:,}")
print(f"Órfãos product  : {orfaos_product:,}")
print(f"Órfãos seller   : {orfaos_seller:,}")
print(f"Órfãos date     : {orfaos_date:,}")

#4.5.2 - Integração com a Dimensão de Data

Após a validação da integridade referencial, a tabela fato é enriquecida com a chave substituta date_key da dimensão de tempo.

O relacionamento é realizado entre a data de compra do pedido e a dimensão dim_date, permitindo análises temporais padronizadas no modelo dimensional.

In [ ]:
fact_order_items_gold = (
    fact_order_items.alias("f")
    .join(
        dim_date
        .select("date", "date_key")
        .alias("d"),
        F.col("f.order_purchase_date") == F.col("d.date"),
        "left"
    )
    .select(
        # Chaves da fato
        F.col("f.order_id"),
        F.col("f.order_item_id"),

        # Chaves dimensionais
        F.col("f.customer_id"),
        F.col("f.product_id"),
        F.col("f.seller_id"),
        F.col("d.date_key").alias("date_key"),

        # Informações operacionais
        F.col("f.order_status"),
        F.col("f.shipping_limit_date"),

        # Métricas
        F.col("f.price"),
        F.col("f.freight_value"),
        F.col("f.item_total_value"),
        F.col("f.delivery_time_days"),
        F.col("f.delivery_delay_days"),
        F.col("f.flag_late_delivery")
    )
)

print("✓ fact_order_items_gold construída.")

In [ ]:
#4.5.3 — Validação final antes da gravação

display(
    fact_order_items_gold.select(
        F.count("*").alias("registros"),

        F.countDistinct(
            "order_id",
            "order_item_id"
        ).alias("chaves_distintas"),

        F.sum(
            F.when(
                F.col("date_key").isNull(),
                1
            ).otherwise(0)
        ).alias("date_key_nula"),

        F.round(
            F.sum("price"),
            2
        ).alias("valor_produtos"),

        F.round(
            F.sum("freight_value"),
            2
        ).alias("valor_frete"),

        F.round(
            F.sum("item_total_value"),
            2
        ).alias("valor_total")
    )
)

#4.5.4 - Validação das Métricas Financeiras

Como controle de qualidade da tabela fato, é realizada a reconciliação entre o valor total dos itens e a soma dos valores de produto e frete.

Essa validação assegura que as métricas derivadas mantenham consistência matemática antes da persistência da tabela na camada Gold.

In [ ]:
validacao_financeira = (
    fact_order_items_gold
    .agg(
        F.round(F.sum("price"), 2).alias("total_produtos"),
        F.round(F.sum("freight_value"), 2).alias("total_frete"),
        F.round(F.sum("item_total_value"), 2).alias("total_itens")
    )
    .withColumn(
        "total_calculado",
        F.round(
            F.col("total_produtos") + F.col("total_frete"),
            2
        )
    )
    .withColumn(
        "diferenca",
        F.round(
            F.col("total_itens") - F.col("total_calculado"),
            2
        )
    )
)

display(validacao_financeira)

In [ ]:
resultado_financeiro = validacao_financeira.first()

diferenca_financeira = abs(
    float(resultado_financeiro["diferenca"])
)

if diferenca_financeira > 0.01:
    raise Exception(
        f"Falha na validação financeira da fato. "
        f"Diferença encontrada: {diferenca_financeira}"
    )

print("✓ Validação financeira concluída com sucesso.")
print(f"✓ Diferença encontrada: {diferenca_financeira:.2f}")

#4.5.5 - Persistência da Fato de Vendas

Após as validações de granularidade, integridade referencial e consistência financeira, a tabela fato é persistida na camada Gold.

A fact_order_items possui granularidade de um registro por item de pedido e concentra as principais chaves dimensionais e métricas necessárias às análises comerciais e logísticas.

In [ ]:
salvar_gold(
    fact_order_items_gold,
    "fact_order_items"
)

In [ ]:
fact_salva = spark.table(
    f"{CATALOG}.{GOLD}.fact_order_items"
)

registros_dataframe = fact_order_items_gold.count()
registros_gold = fact_salva.count()

print(f"Registros DataFrame : {registros_dataframe:,}")
print(f"Registros Gold      : {registros_gold:,}")
print(f"Diferença           : {registros_dataframe - registros_gold:,}")

#4.6 - Construção da Fato de Pagamentos

A tabela fact_payments representa os eventos financeiros associados aos pedidos, mantendo a granularidade original da entidade de pagamentos.

Cada registro corresponde a uma ocorrência de pagamento identificada pela combinação entre order_id e payment_sequential.

A separação entre pagamentos e itens de pedido evita multiplicações decorrentes de relacionamentos entre entidades com granularidades distintas, preservando a consistência das métricas financeiras.

In [ ]:
total_payments = payments.count()

payment_keys = (
    payments
    .select(
        "order_id",
        "payment_sequential"
    )
    .distinct()
    .count()
)

print(f"Registros payments       : {total_payments:,}")
print(f"Chaves distintas         : {payment_keys:,}")
print(f"Duplicidades             : {total_payments - payment_keys:,}")

In [ ]:
pedidos_com_pagamento = (
    payments
    .select("order_id")
    .distinct()
    .count()
)

orfaos_orders = (
    payments
    .select("order_id")
    .distinct()
    .join(
        orders.select("order_id").distinct(),
        "order_id",
        "left_anti"
    )
    .count()
)

print(f"Pedidos com pagamento : {pedidos_com_pagamento:,}")
print(f"Order IDs órfãos       : {orfaos_orders:,}")

In [ ]:
display(
    payments
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("quantidade"),
        F.countDistinct("order_id").alias("pedidos"),
        F.round(
            F.sum("payment_value"),
            2
        ).alias("valor_total")
    )
    .orderBy(F.desc("valor_total"))
)

#4.6.1 - Modelagem da Fato de Pagamentos

A tabela fact_payments é construída na granularidade de ocorrência de pagamento, sendo cada registro identificado pela combinação entre order_id e payment_sequential.

A fato preserva os atributos financeiros provenientes da camada Silver e incorpora as chaves de cliente e tempo por meio da associação com os pedidos.

A manutenção dos pagamentos em uma estrutura independente da fato de itens evita a multiplicação de valores provocada pelas diferentes granularidades das entidades.

In [ ]:
fact_payments = (
    payments.alias("p")
    .join(
        orders.alias("o"),
        F.col("p.order_id") == F.col("o.order_id"),
        "left"
    )
    .join(
        dim_date
        .select("date", "date_key")
        .alias("d"),
        F.to_date(F.col("o.order_purchase_timestamp")) == F.col("d.date"),
        "left"
    )
    .select(
        # Chave degenerada / granularidade
        F.col("p.order_id"),
        F.col("p.payment_sequential"),

        # Chaves dimensionais
        F.col("o.customer_id"),
        F.col("d.date_key"),

        # Atributos do pagamento
        F.col("p.payment_type"),
        F.col("p.payment_installments"),

        # Métrica
        F.col("p.payment_value")
            .cast("double")
            .alias("payment_value")
    )
)

print("✓ fact_payments construída.")

In [ ]:
#4.6.2 — Validação antes de persistir
validacao_payments = (
    fact_payments
    .agg(
        F.count("*").alias("registros"),

        F.countDistinct(
            "order_id",
            "payment_sequential"
        ).alias("chaves_distintas"),

        F.sum(
            F.when(
                F.col("customer_id").isNull(),
                1
            ).otherwise(0)
        ).alias("customer_id_nulo"),

        F.sum(
            F.when(
                F.col("date_key").isNull(),
                1
            ).otherwise(0)
        ).alias("date_key_nula"),

        F.round(
            F.sum("payment_value"),
            2
        ).alias("valor_pagamentos")
    )
)

display(validacao_payments)


In [ ]:
orfaos_customer_payments = (
    fact_payments
    .select("customer_id")
    .distinct()
    .join(
        dim_customer
        .select("customer_id")
        .distinct(),
        "customer_id",
        "left_anti"
    )
    .count()
)

print(
    f"Customers órfãos na fact_payments: "
    f"{orfaos_customer_payments:,}"
)

#4.6.3 - Persistência da fato de pagamentos

Após a construção da fato, foram realizadas validações de granularidade e integridade referencial. A combinação order_id e payment_sequential apresentou unicidade para os 103.886 registros, não sendo identificadas chaves de clientes ou datas sem correspondência nas respectivas dimensões.

O valor financeiro total registrado na fato corresponde a R$ 16.008.872,12. Após as validações, a estrutura é persistida na camada Gold em formato Delta.

In [ ]:
salvar_gold(
    fact_payments,
    "fact_payments"
)

In [ ]:
fact_payments_salva = spark.table(
    f"{CATALOG}.{GOLD}.fact_payments"
)

registros_dataframe = fact_payments.count()
registros_gold = fact_payments_salva.count()

print(f"Registros DataFrame : {registros_dataframe:,}")
print(f"Registros Gold      : {registros_gold:,}")
print(f"Diferença           : {registros_dataframe - registros_gold:,}")

#4.7 — Dimensão de Pedidos

A dimensão de pedidos é construída a partir da entidade orders da camada Silver, preservando a granularidade de um registro por pedido.

A estrutura concentra atributos operacionais e temporais relacionados ao ciclo de vida do pedido, evitando sua replicação desnecessária nas tabelas fato. A dimensão também fornece uma entidade comum para análises envolvendo itens e pagamentos, cujas tabelas fato possuem granularidades distintas.

As métricas financeiras permanecem nas respectivas tabelas fato, enquanto a dimensão mantém informações descritivas e indicadores associados ao pedido.

In [ ]:
#4.7.1 - Construção da Dimensão

dim_order = (
    orders
    .select(
        "order_id",
        "customer_id",
        "order_status",

        # Datas do ciclo do pedido
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",

        # Métricas e indicadores tratados na Silver
        "delivery_time_days",
        "delivery_delay_days",
        "flag_late_delivery"
    )
)

print("✓ dim_order construída.")


In [ ]:
total_orders = orders.count()

order_keys = (
    dim_order
    .select("order_id")
    .distinct()
    .count()
)

total_dim_order = dim_order.count()

print(f"Registros Silver orders : {total_orders:,}")
print(f"Registros dim_order     : {total_dim_order:,}")
print(f"Order IDs distintos     : {order_keys:,}")
print(f"Duplicidades            : {total_dim_order - order_keys:,}")

In [ ]:
# ============================================================
# 4.7.2 - Integridade da dim_order com dim_customer
# ============================================================

customer_id_nulos = dim_order.filter(F.col("customer_id").isNull()).count()

orders_sem_customer = (
    dim_order
    .select("customer_id")
    .filter(F.col("customer_id").isNotNull())
    .distinct()
    .join(
        dim_customer.select("customer_id").distinct(),
        "customer_id",
        "left_anti"
    )
    .count()
)

print(f"customer_id nulos      : {customer_id_nulos:,}")
print(f"Customers órfãos       : {orders_sem_customer:,}")

if customer_id_nulos > 0 or orders_sem_customer > 0:
    raise Exception("Falha de integridade entre dim_order e dim_customer.")

print("✓ Integridade dim_order -> dim_customer validada.")


In [ ]:
#4.7.2 — Validação dos atributos da dim_order

display(
    dim_order
    .groupBy("order_status")
    .agg(
        F.count("*").alias("quantidade"),
        F.sum(
            F.when(F.col("flag_late_delivery") == 1, 1)
             .otherwise(0)
        ).alias("entregas_atrasadas"),
        F.round(
            F.avg("delivery_time_days"),
            2
        ).alias("tempo_medio_entrega"),
        F.round(
            F.avg("delivery_delay_days"),
            2
        ).alias("atraso_medio")
    )
    .orderBy(F.desc("quantidade"))
)

In [ ]:
display(
    dim_order.select(
        F.count("*").alias("registros"),

        F.sum(
            F.when(
                F.col("order_status").isNull(),
                1
            ).otherwise(0)
        ).alias("status_nulo"),

        F.sum(
            F.when(
                F.col("order_purchase_timestamp").isNull(),
                1
            ).otherwise(0)
        ).alias("data_compra_nula"),

        F.sum(
            F.when(
                F.col("delivery_time_days").isNull(),
                1
            ).otherwise(0)
        ).alias("tempo_entrega_nulo"),

        F.sum(
            F.when(
                F.col("delivery_delay_days").isNull(),
                1
            ).otherwise(0)
        ).alias("delay_nulo")
    )
)

In [ ]:
#4.7.3 — Validação final da dim_order

delivered_sem_data = (
    dim_order
    .filter(
        (F.col("order_status") == "delivered") &
        (F.col("order_delivered_customer_date").isNull())
    )
    .count()
)

print(f"Pedidos delivered sem data de entrega: {delivered_sem_data:,}")

In [ ]:
inconsistencia_flag = (
    dim_order
    .filter(
        (
            (F.col("delivery_delay_days") > 0) &
            (F.col("flag_late_delivery") != 1)
        )
        |
        (
            (F.col("delivery_delay_days") <= 0) &
            (F.col("flag_late_delivery") == 1)
        )
    )
    .count()
)

print(f"Inconsistências na flag de atraso: {inconsistencia_flag:,}")

In [ ]:
#4.7.4 — Diagnóstico da flag de atraso

diagnostico_flag = (
    dim_order
    .groupBy(
        "flag_late_delivery"
    )
    .agg(
        F.count("*").alias("registros"),

        F.sum(
            F.when(F.col("delivery_delay_days") > 0, 1)
             .otherwise(0)
        ).alias("delay_positivo"),

        F.sum(
            F.when(F.col("delivery_delay_days") == 0, 1)
             .otherwise(0)
        ).alias("delay_zero"),

        F.sum(
            F.when(F.col("delivery_delay_days") < 0, 1)
             .otherwise(0)
        ).alias("delay_negativo"),

        F.sum(
            F.when(F.col("delivery_delay_days").isNull(), 1)
             .otherwise(0)
        ).alias("delay_nulo")
    )
)

display(diagnostico_flag)

In [ ]:
display(
    dim_order
    .filter(
        (
            (F.col("delivery_delay_days") > 0) &
            (F.col("flag_late_delivery") != 1)
        )
        |
        (
            (F.col("delivery_delay_days") <= 0) &
            (F.col("flag_late_delivery") == 1)
        )
    )
    .select(
        "order_id",
        "order_status",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay_days",
        "flag_late_delivery"
    )
    .orderBy(F.desc("delivery_delay_days"))
    .limit(30)
)

In [ ]:
salvar_gold(
    dim_order,
    "dim_order"
)

In [ ]:
dim_order_salva = spark.table(
    f"{CATALOG}.{GOLD}.dim_order"
)

print(f"Registros DataFrame : {dim_order.count():,}")
print(f"Registros Gold      : {dim_order_salva.count():,}")
print(f"Diferença           : {dim_order.count() - dim_order_salva.count():,}")

In [ ]:
inconsistencia_gold = (
    dim_order_salva
    .filter(
        (
            (F.col("delivery_delay_days") > 0) &
            (F.col("flag_late_delivery") != 1)
        )
        |
        (
            (F.col("delivery_delay_days") <= 0) &
            (F.col("flag_late_delivery") == 1)
        )
    )
    .count()
)

print(
    f"Inconsistências na dim_order salva na Gold: "
    f"{inconsistencia_gold:,}"
)

## 4.8 - Reprocessamento da fact_order_items após correção da regra de atraso

Após a correção de `flag_late_delivery` na camada Silver e a atualização da `dim_order`, a fato de itens deve ser reconstruída para propagar a regra corrigida. Antes da persistência, são validadas a granularidade, a chave de data e a consistência entre `delivery_delay_days` e `flag_late_delivery`.


In [ ]:
# Recarrega orders da Silver para garantir o uso da versão corrigida
orders = silver_table("orders")

fact_order_items = (
    items.alias("i")
    .join(orders.alias("o"), F.col("i.order_id") == F.col("o.order_id"), "left")
    .select(
        F.col("i.order_id"),
        F.col("i.order_item_id"),
        F.col("o.customer_id"),
        F.col("i.product_id"),
        F.col("i.seller_id"),
        F.to_date(F.col("o.order_purchase_timestamp")).alias("order_purchase_date"),
        F.col("o.order_status"),
        F.col("i.shipping_limit_date"),
        F.col("i.price"),
        F.col("i.freight_value"),
        F.col("i.item_total_value"),
        F.col("o.delivery_time_days"),
        F.col("o.delivery_delay_days"),
        F.col("o.flag_late_delivery")
    )
)

fact_order_items_gold = (
    fact_order_items.alias("f")
    .join(dim_date.select("date", "date_key").alias("d"),
          F.col("f.order_purchase_date") == F.col("d.date"), "left")
    .select(
        F.col("f.order_id"), F.col("f.order_item_id"),
        F.col("f.customer_id"), F.col("f.product_id"), F.col("f.seller_id"),
        F.col("d.date_key").alias("date_key"),
        F.col("f.order_status"), F.col("f.shipping_limit_date"),
        F.col("f.price"), F.col("f.freight_value"), F.col("f.item_total_value"),
        F.col("f.delivery_time_days"), F.col("f.delivery_delay_days"),
        F.col("f.flag_late_delivery")
    )
)

print("✓ fact_order_items_gold reconstruída com a regra de atraso corrigida.")


In [ ]:
# Validação da regra de atraso na fato reconstruída
inconsistencia_flag_fact = (
    fact_order_items_gold
    .filter(
        ((F.col("delivery_delay_days") > 0) & (F.col("flag_late_delivery") != 1)) |
        ((F.col("delivery_delay_days") <= 0) & (F.col("flag_late_delivery") == 1)) |
        (F.col("flag_late_delivery").isNull())
    )
    .count()
)

display(
    fact_order_items_gold
    .groupBy("flag_late_delivery")
    .agg(
        F.count("*").alias("registros"),
        F.sum(F.when(F.col("delivery_delay_days") > 0, 1).otherwise(0)).alias("delay_positivo"),
        F.sum(F.when(F.col("delivery_delay_days") == 0, 1).otherwise(0)).alias("delay_zero"),
        F.sum(F.when(F.col("delivery_delay_days") < 0, 1).otherwise(0)).alias("delay_negativo"),
        F.sum(F.when(F.col("delivery_delay_days").isNull(), 1).otherwise(0)).alias("delay_nulo")
    )
)

print(f"Inconsistências na flag da fact_order_items: {inconsistencia_flag_fact:,}")

if inconsistencia_flag_fact != 0:
    raise Exception(f"Falha na validação da fact_order_items: {inconsistencia_flag_fact:,} inconsistências na flag de atraso.")


In [ ]:
# Persistência somente após validação bem-sucedida
salvar_gold(fact_order_items_gold, "fact_order_items")

fact_order_items_salva = spark.table(f"{CATALOG}.{GOLD}.fact_order_items")

registros_dataframe = fact_order_items_gold.count()
registros_gold = fact_order_items_salva.count()
chaves_distintas = fact_order_items_salva.select("order_id", "order_item_id").distinct().count()

print(f"Registros DataFrame : {registros_dataframe:,}")
print(f"Registros Gold      : {registros_gold:,}")
print(f"Chaves distintas    : {chaves_distintas:,}")
print(f"Diferença           : {registros_dataframe - registros_gold:,}")
print(f"Duplicidades        : {registros_gold - chaves_distintas:,}")
